# 02 Evaluation

Purpose: Evaluate all five scoring variants on the test set, run alpha sensitivity for local DNDS, and save summary artifacts.

Inputs: `dataset/CVPR_2024_dataset_Test/`, `dataset_text/test.csv`, `models/mobilenetv3_best.pth`, `text_models/distilbert_best.pth`, populated `chroma_db/`.

Outputs: `results/phase2/evaluation_results.json`, `figures/phase2/scoring_comparison.png`, `figures/phase2/alpha_sensitivity.png`, `figures/phase2/confusion_matrices_phase2.png`, `figures/phase2/phase2_vs_phase1.png`.

In [ ]:
import importlib
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

from src.phase2.config import get_phase2_config
from src.phase2.data_utils import build_records_from_csv
from src.phase2.db_client import (
    get_class_counts,
    get_image_collection,
    get_persistent_client,
    get_text_collection,
)
import src.phase2.evaluation as phase2_evaluation

importlib.reload(phase2_evaluation)
from src.phase2.evaluation import (
    evaluate_variant,
    save_alpha_sweep_csv,
    save_metrics_summary_csv,
    save_predictions_csv,
    save_results,
)
from src.phase2.imbalance import infer_class_groups
from src.phase2.scoring import global_dnds, idw, local_dnds, majority_vote, traditional
from src.phase2.traditional import load_phase1_traditional_components
from src.phase2.visualization import (
    plot_alpha_sensitivity,
    plot_confusion_matrices,
    plot_phase2_vs_phase1,
    plot_scoring_comparison,
)

CONFIG = get_phase2_config()

REPO_ROOT = Path("../..").resolve()
TEST_DIR = REPO_ROOT / "dataset" / "CVPR_2024_dataset_Test"
TEST_CSV = REPO_ROOT / "dataset_text" / "test.csv"
RESULTS_PATH = REPO_ROOT / "results" / "phase2" / "evaluation_results.json"
METRICS_CSV_PATH = REPO_ROOT / "results" / "phase2" / "metrics_summary.csv"
ALPHA_CSV_PATH = REPO_ROOT / "results" / "phase2" / "alpha_sweep.csv"
PREDICTIONS_CSV_PATH = (
    REPO_ROOT / "results" / "phase2" / "scoring_variants_predictions.csv"
)
FIGURES_DIR = REPO_ROOT / "figures" / "phase2"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not TEST_DIR.exists() or not TEST_CSV.exists():
    raise FileNotFoundError(
        "Test dataset missing. Expected dataset and dataset_text paths in repo root."
    )

In [ ]:
test_samples, missing_examples, total_rows = build_records_from_csv(
    csv_path=TEST_CSV,
    split_dir=TEST_DIR,
    text_column="text",
    label_column="label",
    text_key="text",
)

if missing_examples:
    print("Skipped test rows with missing image files (up to 10 shown):")
    for item in missing_examples:
        print(f"  - {item}")

if not test_samples:
    raise RuntimeError("No test samples found after image path resolution.")

print(f"Test samples available for evaluation: {len(test_samples)} / {total_rows}")

client = get_persistent_client(str(REPO_ROOT / "chroma_db"))
image_collection = get_image_collection(client)
text_collection = get_text_collection(client)

image_class_counts = get_class_counts(image_collection)
majority_classes, minority_classes = infer_class_groups(
    image_class_counts, threshold=float(CONFIG["majority_threshold"])
)
CONFIG["majority_classes"] = majority_classes
CONFIG["minority_classes"] = minority_classes
print(
    f"Dynamic class grouping -> majority: {majority_classes}, minority: {minority_classes}"
)

variant_fns = {
    "majority_vote": majority_vote,
    "idw": idw,
    "global_dnds": global_dnds,
    "local_dnds": local_dnds,
}

In [ ]:
results = {"variants": {}, "alpha_sweep": {"alphas": [], "macro_f1": []}}

for name, fn in variant_fns.items():
    metrics = evaluate_variant(
        fn, test_samples, image_collection, text_collection, CONFIG
    )
    results["variants"][name] = metrics

# Alpha sweep for local DNDS
for alpha in CONFIG["alpha_sweep"]:
    metrics = evaluate_variant(
        local_dnds, test_samples, image_collection, text_collection, CONFIG, alpha=alpha
    )
    results["alpha_sweep"]["alphas"].append(alpha)
    results["alpha_sweep"]["macro_f1"].append(metrics["macro_f1"])

In [ ]:
# Traditional baseline
image_ckpt = REPO_ROOT / "models" / "mobilenetv3_best.pth"
text_ckpt = REPO_ROOT / "text_models" / "distilbert_best.pth"

if image_ckpt.exists() and text_ckpt.exists():
    components = load_phase1_traditional_components(
        image_checkpoint_path=image_ckpt,
        text_checkpoint_path=text_ckpt,
        num_classes=len(CONFIG["class_names"]),
        device="cpu",
    )

    traditional_metrics = evaluate_variant(
        traditional,
        test_samples,
        image_collection,
        text_collection,
        CONFIG,
        score_kwargs=components,
    )
    results["variants"]["traditional"] = traditional_metrics
else:
    print("Traditional checkpoints missing; skipping traditional baseline run.")

In [ ]:
best_alpha_idx = max(
    range(len(results["alpha_sweep"]["macro_f1"])),
    key=lambda i: results["alpha_sweep"]["macro_f1"][i],
)
results["best_alpha"] = results["alpha_sweep"]["alphas"][best_alpha_idx]

save_results(results, str(RESULTS_PATH))
save_metrics_summary_csv(results["variants"], str(METRICS_CSV_PATH))
save_alpha_sweep_csv(results["alpha_sweep"], str(ALPHA_CSV_PATH))
save_predictions_csv(results["variants"], str(PREDICTIONS_CSV_PATH))

plot_scoring_comparison(results, str(FIGURES_DIR / "scoring_comparison.png"))
plot_alpha_sensitivity(
    results["alpha_sweep"], str(FIGURES_DIR / "alpha_sensitivity.png")
)
plot_confusion_matrices(
    results, CONFIG["class_names"], str(FIGURES_DIR / "confusion_matrices_phase2.png")
)
phase1_macro_f1 = 0.8177
best_phase2 = max(
    v.get("macro_f1", 0.0) for k, v in results["variants"].items() if k != "traditional"
)
plot_phase2_vs_phase1(
    best_phase2, phase1_macro_f1, str(FIGURES_DIR / "phase2_vs_phase1.png")
)

print(f"Saved JSON results: {RESULTS_PATH}")
print(f"Saved CSV metrics: {METRICS_CSV_PATH}")
print(f"Saved CSV alpha sweep: {ALPHA_CSV_PATH}")
print(f"Saved CSV predictions: {PREDICTIONS_CSV_PATH}")

In [ ]:
print("=" * 60)
print("PHASE 2 RAC EXPERIMENT - RESULTS SUMMARY")
print("=" * 60)
print("Variant              | Accuracy | Macro F1 | TTR F1 | Latency")
print("---------------------|----------|----------|--------|--------")
for name, metrics in results["variants"].items():
    ttr_f1 = metrics.get("per_class_f1", {}).get("TTR", 0.0)
    print(
        f"{name:<20} | {metrics['accuracy']*100:6.2f}% | {metrics['macro_f1']:.4f} | {ttr_f1:.4f} | {metrics['latency_ms_per_sample']:.2f} ms"
    )
print("=" * 60)
print(f"Best alpha (local DNDS): {results['best_alpha']}")